# Lesson 2: Many inputs and chains of functions. Partial derivatives and the chain rule

## The old ideas

- **Partial derivatives (Euler, Lagrange, 1700s):** when a quantity depends on *several* things (the temperature of a metal plate depends on where you stand: east-west *and* north-south), you can ask how it changes if you move along just one direction while holding the others still. Each of these "one direction at a time" slopes is a *partial* derivative.
- **The chain rule (Leibniz, late 1600s):** when one quantity feeds into another, which feeds into another, the rates of change **multiply**. If a car's wheel turns twice per engine turn, and the road moves 3 metres per wheel turn, then the road moves 6 metres per engine turn.

## What they become in machine learning

- A model has many weights, not one. The **gradient** is simply the list of partial derivatives, one per weight, telling you how the error responds to each weight.
- A neural network is a chain: input goes through layer 1, whose output goes through layer 2, and so on, ending in an error number. The chain rule is how you find the slope of the final error with respect to a weight buried in an early layer. **Backpropagation is the chain rule, organised so a computer can do it efficiently** (Lesson 14). By the end of this lesson you'll have built the 1-dimensional version yourself.

**By the end of this lesson you can:** compute a gradient numerically, derive one by hand using the chain rule, and build a small class that applies the chain rule to any chain of functions.

---
## Step 0: Setup from Lesson 1

*Setup cell: run this as-is (it's code you already wrote in an earlier lesson).*

In [ ]:
def slope_by_wiggling(function, point, gap=0.0001):
    return (function(point + gap) - function(point)) / gap

---
## Step 1: One input at a time (partial derivatives)

Here's a function of **two** inputs, a long narrow valley: `valley(x, y) = x² + 3y²`. We'll pass its inputs in as one **list** `[x, y]`, because a model's weights will arrive as a list. Type:

```python
def valley(point):
    x_value, y_value = point
    return x_value ** 2 + 3 * y_value ** 2

print(valley([1, 2]))
print(valley([0, 0]))
```

In [ ]:
# type it here

**Expected output:**
```text
13
0
```

`x_value, y_value = point` is **unpacking**: Python splits a two-item list into two named variables. At `[1, 2]`: `1 + 3*4 = 13`.

A **partial derivative** asks: wiggle *only one* input, hold the others still, and measure how the output responds. Before writing it, one detail about how Python stores lists that will bite you otherwise. Type this experiment:

```python
original = [1, 2]
alias = original          # NOT a copy
alias[0] = 99
print(original)           # what do you expect?

original = [1, 2]
real_copy = list(original)   # a genuine copy
real_copy[0] = 99
print(original)
```

In [ ]:
# type it here

**Expected output:**
```text
[99, 2]
[1, 2]
```

A list variable doesn't hold the numbers, it holds a *reference* to the one list in memory. `alias = original` gives the same list a second name, so changing one changes both. `list(original)` builds a new list. We need a copy so wiggling one input doesn't corrupt the original point. Now the partial derivative:

```python
def partial_derivative(function, point, which_input, gap=0.0001):
    wiggled_point = list(point)
    wiggled_point[which_input] = wiggled_point[which_input] + gap
    return (function(wiggled_point) - function(point)) / gap

print(partial_derivative(valley, [1, 2], 0))
print(partial_derivative(valley, [1, 2], 1))
```

In [ ]:
# type it here

**Expected output:**
```text
2.000100000003613
12.000300000032382
```

**Signature:** `function` takes a list and returns a number; `point` is the list where we measure; `which_input` is the **index** of the input to wiggle (0 = first, 1 = second). It's the same wiggle as Lesson 1, applied to one position.

In a textbook the partial slope with respect to `x` is written `∂f/∂x` (the curly "∂" instead of "d" is the signal "there are other inputs I'm holding still"). Check by hand: treat `y` as a constant, so `∂/∂x (x² + 3y²) = 2x` (the `3y²` doesn't change with x, so contributes 0). And `∂/∂y = 6y`. At `[1, 2]`: `2·1 = 2` and `6·2 = 12`. The numbers above should be about 2 and 12. ✓

**Try this (new cell):** call `partial_derivative(valley, [3, 0.5], 0)` and `(valley, [3, 0.5], 1)`. Predict both by hand first. Which direction is steeper at that point?

---
## Step 2: The gradient is the list of partial derivatives

```python
def gradient(function, point):
    return [partial_derivative(function, point, index) for index in range(len(point))]

print(gradient(valley, [1, 2]))
print(gradient(valley, [-3, 1]))
print(gradient(valley, [0, 0]))
```

In [ ]:
# type it here

**Expected output:**
```text
[2.000100000003613, 12.000300000032382]
[-5.99990000001327, 6.000299999993075]
[9.999999999999999e-05, 0.00030000000000000003]
```

The line inside is a **list comprehension**: "for each `index` from 0 up to the number of inputs, compute that partial derivative, and collect the results in a list." It's a loop that builds a list in one line. Written out long:

```python
slopes = []
for index in range(len(point)):
    slopes.append(partial_derivative(function, point, index))
return slopes
```

In a textbook this list is written `∇f` ("nabla f" or "grad f"). Two facts make it the most important vector in machine learning:

1. **It points in the direction of steepest *uphill*.** Its length is how steep that slope is.
2. So the **negative** gradient points steepest *downhill*. At `[0, 0]` (the bottom of the valley) it's `[0, 0]`: flat, nowhere to go.

Test fact 2: step a little *against* the gradient and see whether the value falls.

```python
point = [1, 2]
slopes = gradient(valley, point)
step_size = 0.01

stepped_point = [
    point[0] - step_size * slopes[0],
    point[1] - step_size * slopes[1],
]

print("before:", valley(point))
print("after: ", valley(stepped_point))
print(stepped_point)
```

In [ ]:
# type it here

**Expected output:**
```text
before: 13
after:  11.563564200024276
[0.9799989999999639, 1.8799969999996762]
```

The value fell from 13 to about 11.56. The new point is `[0.98, 1.88]`. Notice that `y` moved *six times* as far as `x` because the valley is six times steeper in the `y` direction. The gradient doesn't just say "downhill", it says how hard each input pulls. That's exactly the update used to train every neural network.

---
## Step 3: The chain rule

Next, functions *of* functions. A circular ink blot is spreading: its radius after `t` seconds is `radius(t) = 2t` centimetres, and its area is `area(r) = π r²`. How fast is the **area** growing at t = 3 seconds?

Break it into links: time → radius → area. Type:

```python
import math

def radius(seconds):
    return 2 * seconds

def area(radius_value):
    return math.pi * radius_value ** 2

def area_at_time(seconds):
    return area(radius(seconds))

print(area_at_time(3))
print(slope_by_wiggling(area_at_time, 3))
```

In [ ]:
# type it here

**Expected output:**
```text
113.09733552923255
75.39948032331267
```

The brute-force wiggle gives about 75.4 square centimetres per second. Now the chain-rule route. Reason it through with tiny changes:

- Let time increase by a tiny `dt`. The radius changes by `radius_slope × dt` (here `2 × dt`).
- That tiny radius change changes the area by `area_slope × (change in radius)`, where `area_slope = 2πr` (power rule on `πr²`).
- Put together: change in area `= area_slope × radius_slope × dt`.

So **the slopes multiply**:

```
d(area)/dt  =  d(area)/d(radius)  ×  d(radius)/dt        (Leibniz's notation makes it look like fractions cancelling)
```

At t = 3: radius = 6, so `area_slope = 2π·6 = 37.699...` and `radius_slope = 2`. Product: `75.398...`. Verify in code:

```python
current_time = 3
current_radius = radius(current_time)

area_slope = 2 * math.pi * current_radius
radius_slope = 2

print(area_slope * radius_slope)
```

In [ ]:
# type it here

**Expected output:**
```text
75.39822368615503
```

The chain rule gives 75.398..., and the wiggle gave 75.4. Same answer, found by multiplying two *simple* slopes instead of wiggling the whole composite.

**Why this matters:** you never need to differentiate the whole messy composite, only each link's own simple slope, evaluated where that link was actually working. Then multiply.

---
## Step 4: A chain of three

Chains can be as long as you like: `y = cube( square( 2x + 1 ) )`. Links: `x → u = 2x+1 → v = u² → y = v³`. Their local slopes: `du/dx = 2`, `dv/du = 2u`, `dy/dv = 3v²`. By the chain rule, `dy/dx = 2 × 2u × 3v²`. At `x = 1`: `u = 3`, `v = 9`, so `dy/dx = 2 × 6 × 243 = 2916`.

(Sanity check by algebra: `y = (2x+1)⁶`, so `dy/dx = 6(2x+1)⁵ × 2 = 12 × 243 = 2916`. ✓)

Now build a **class** that does this for *any* chain. Each link is a pair: the function, and its own simple slope function. The object stores the list of links. It evaluates forward, **remembering the input each link received** (the slopes must be evaluated where each link was working), then multiplies the local slopes. Type this class:

```python
class FunctionChain:
    def __init__(self):
        self.links = []

    def add_link(self, function, slope_function):
        self.links.append((function, slope_function))

    def forward(self, starting_value):
        inputs_seen = []
        current_value = starting_value
        for function, slope_function in self.links:
            inputs_seen.append(current_value)
            current_value = function(current_value)
        return current_value, inputs_seen

    def derivative_at(self, starting_value):
        final_value, inputs_seen = self.forward(starting_value)
        total_slope = 1.0
        for (function, slope_function), input_to_link in zip(self.links, inputs_seen):
            total_slope = total_slope * slope_function(input_to_link)
        return total_slope
```

In [ ]:
# type the class here

**How it stores data:** `self.links` is a list of `(function, slope_function)` pairs. `forward` loops through the links in order, appending each link's *input* to `inputs_seen` before applying the link. `derivative_at` calls `forward` for those remembered inputs, then multiplies each link's `slope_function(input)`. `zip(a, b)` walks two lists in step, pairing item 0 with item 0, item 1 with item 1, and so on.

Use it on the chain from above:

```python
chain = FunctionChain()
chain.add_link(lambda x: 2 * x + 1,  lambda x: 2)
chain.add_link(lambda u: u ** 2,     lambda u: 2 * u)
chain.add_link(lambda v: v ** 3,     lambda v: 3 * v ** 2)

final_value, inputs_seen = chain.forward(1)
print(final_value, inputs_seen)
print(chain.derivative_at(1))

def whole_chain(x):
    return chain.forward(x)[0]

print(slope_by_wiggling(whole_chain, 1))
```

In [ ]:
# type it here

**Expected output:**
```text
729 [1, 3, 9]
2916.0
2916.4860432035766
```

`lambda x: 2 * x + 1` is a tiny unnamed function: "take `x`, return `2 * x + 1`". The chain's forward pass gives `729` and the inputs each link saw: `[1, 3, 9]`. The chain-rule slope is exactly `2916.0`; the wiggled one lands near it.

Let's add a method that makes the *mechanism* visible: it prints each link's local slope as it multiplies. **Add this inside the class** (below `derivative_at`, same indent as the others), re-run the class cell, then re-run the chain-building cell above:

**Add this *inside* the class (indented, below the existing methods) in the class cell above, then re-run that cell:**

```python
    def explain(self, starting_value):
        final_value, inputs_seen = self.forward(starting_value)
        running_product = 1.0
        for link_number, ((function, slope_function), input_to_link) in enumerate(zip(self.links, inputs_seen)):
            local_slope = slope_function(input_to_link)
            running_product = running_product * local_slope
            print(f"link {link_number}: input={input_to_link}  local slope={local_slope}  running product={running_product}")
        return running_product
```

```python
chain = FunctionChain()
chain.add_link(lambda x: 2 * x + 1,  lambda x: 2)
chain.add_link(lambda u: u ** 2,     lambda u: 2 * u)
chain.add_link(lambda v: v ** 3,     lambda v: 3 * v ** 2)

chain.explain(1)
```

In [ ]:
# type it here

**Expected output:**
```text
link 0: input=1  local slope=2  running product=2.0
link 1: input=3  local slope=6  running product=12.0
link 2: input=9  local slope=243  running product=2916.0
```

Watch the running product grow `2 → 12 → 2916`. **This is backpropagation in miniature**: a forward pass that remembers values, and local slopes multiplied together. Real backprop multiplies from the *output end backward* (so one pass delivers the slope for every weight at once), but the maths is exactly this chain rule. Lesson 14 builds the backward version.

---
## Step 5: The chain rule on a real model error

Now the pattern behind linear regression. A model predicts `prediction = w·x + b` for one data point (say `x = 3`, true answer `y = 5`). The error is `prediction − y` and the loss is the error squared. Inputs to the function are the **weights** `[w, b]`. Type:

```python
def single_example_loss(weights):
    weight, bias = weights
    prediction = weight * 3 + bias
    error = prediction - 5
    return error ** 2

print(single_example_loss([2, 1]))
print(gradient(single_example_loss, [2, 1]))
```

In [ ]:
# type it here

**Expected output:**
```text
4
[12.0009000000465, 4.00009999999007]
```

At `[2, 1]`: prediction `2·3 + 1 = 7`, error `7 − 5 = 2`, loss `4`. The numerical gradient is about `[12, 4]`. Derive it with the chain rule, links `weight → prediction → error → loss`:

```
∂loss/∂error      = 2 × error            = 2 × 2 = 4     (power rule on error²)
∂error/∂weight    = x                    = 3             (prediction = weight·x + bias; error = prediction − y)
∂error/∂bias      = 1

∂loss/∂weight = (∂loss/∂error) × (∂error/∂weight) = 4 × 3 = 12
∂loss/∂bias   = (∂loss/∂error) × (∂error/∂bias)   = 4 × 1 = 4
```

Exactly the `[12, 4]` the wiggle found. Read it as a sentence: **"the weight's pull on the loss is 2 × error × the input that weight multiplied."** Remember that sentence. It's the entire gradient for linear regression (Lesson 3), and for the output layer of a neural network.

---
## Step 6: Guided exploration: swap the outer function

The squaring step is a choice. A named alternative: **absolute error**, `|error|` (what you get if you don't like big errors being punished so heavily). Copy the Step 5 cell into a new cell, replace `error ** 2` with `abs(error)`, and predict the gradient at `[2, 1]` first. (Hint: chain rule again; the slope of `abs(u)` is `+1` when `u` is positive, `−1` when negative.)

```python
def single_example_absolute_loss(weights):
    weight, bias = weights
    prediction = weight * 3 + bias
    error = prediction - 5
    return abs(error)

print(gradient(single_example_absolute_loss, [2, 1]))
```

In [ ]:
# type it here

**Expected output:**
```text
[3.000000000010772, 0.9999999999976694]
```

Gradient `[3, 1]`: `slope of abs = +1`, times `3` and `1`. Note the size no longer depends on how big the error is. Squared error gives *bigger pulls for bigger mistakes*; absolute error pulls with constant strength.

Now the weak spot. Put the model exactly on the answer (`weight = 1, bias = 2`: prediction 5, error 0) and compute the slope wiggling forward, then backward (negative gap):

```python
on_the_answer = [1, 2]
print(partial_derivative(single_example_absolute_loss, on_the_answer, 0, gap=0.0001))
print(partial_derivative(single_example_absolute_loss, on_the_answer, 0, gap=-0.0001))
```

In [ ]:
# type it here

**Expected output:**
```text
3.00000000000189
-3.00000000000189
```

Forward says `+3`, backward says `−3`: the V-shaped curve has a **kink** at the bottom with no single slope there. Squared error is a smooth bowl with slope 0 at the bottom. That smoothness is a big reason squared error is the default. We'll return to this when deriving least squares, and again when we meet the probability reason (Lesson 9).

---
## Step 7: Challenge

1. **Chain by hand.** `y = sqrt(x² + 1)`, links `x → u = x² + 1 → y = u^0.5`. Write the two local slopes and combine them to get `dy/dx` at `x = 2`. Then check using `FunctionChain` (give it two links) and `slope_by_wiggling`.
2. **Gradient by hand.** `f(x, y) = x·y + y²`. Work out `∂f/∂x` and `∂f/∂y` by hand (treat the other variable as a constant), evaluate at `[3, 2]`, and check with `gradient`.
3. **Descent.** From `[3, 2]`, take one step against the gradient with `step_size = 0.1`. Does `f` go down?

Paper first, then code.

```python
# your attempt here
```

In [ ]:
# type it here

---
## Where this goes next

You now have both halves of the machinery: a slope for every input (the gradient), and a rule for slopes through chains of calculations. Lesson 3 puts them to work on the oldest ML problem there is, **fitting a line through measurements** (least squares, 1805), and finds the answer exactly. Lesson 4 then shows how to find it by walking downhill when no exact formula exists.

Once you're comfortable, one thing to probe: the `explain` output multiplied slopes in the *forward* order. Search "forward-mode vs reverse-mode automatic differentiation" to find out why networks multiply from the *other* end, and why the choice depends on whether you have many inputs and few outputs, or the reverse.

---
# Solutions to the Step 7 challenge (try it first)

**1.** Local slopes: `du/dx = 2x`, `dy/du = 0.5 · u^(−0.5)`. At `x = 2`: `u = 5`, so `dy/dx = 0.5 · 5^(−0.5) · 4 = 2/√5 ≈ 0.8944`.

```python
chain_for_square_root = FunctionChain()
chain_for_square_root.add_link(lambda x: x ** 2 + 1, lambda x: 2 * x)
chain_for_square_root.add_link(lambda u: u ** 0.5,   lambda u: 0.5 * u ** -0.5)

print(chain_for_square_root.derivative_at(2))
print(slope_by_wiggling(lambda x: chain_for_square_root.forward(x)[0], 2))
print(2 / math.sqrt(5))
```

**Output:**
```text
0.8944271909999159
0.8944316629566629
0.8944271909999159
```

**2.** `∂f/∂x = y` (the `y²` term has no `x`); `∂f/∂y = x + 2y` (power rule on `y²` plus the `x·y` term). At `[3, 2]`: `[2, 7]`.

```python
def two_input_function(point):
    x_value, y_value = point
    return x_value * y_value + y_value ** 2

print(gradient(two_input_function, [3, 2]))
```

**Output:**
```text
[1.9999999999953388, 7.000100000027487]
```

**3.** One downhill step, `[3, 2] − 0.1 × [2, 7] = [2.8, 1.3]`:

```python
slopes = [2, 7]
stepped_point = [3 - 0.1 * slopes[0], 2 - 0.1 * slopes[1]]
print(two_input_function([3, 2]), "->", two_input_function(stepped_point))
```

**Output:**
```text
10 -> 5.329999999999998
```

Value fell from 10 to about 5.33. Walking against the gradient reduces the function: the idea behind training, made rigorous in Lesson 4.